# JIT Benchmark for Training Path

This notebook benchmarks the current training path with and without JIT helpers used in `train.py`.

Methodology (from JAX docs via Context7):
- Warm up JIT before timed runs.
- Use `block_until_ready()` so timing includes actual compute time.
- Keep tensor shapes fixed across runs for fair comparison.

In [3]:
import time
from functools import partial
from pathlib import Path
import os
import sys

import jax
import jax.numpy as jnp
import pandas as pd

from jax import random

project_root = Path.cwd()
if not (project_root / "config.py").exists() and (project_root.parent / "config.py").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

os.chdir(project_root)

from config import Config as config
from model import NGCTransformer
from data_preprocess.data_loader import DataLoader
from ngclearn.utils.metric_utils import measure_CatNLL

print('Project root:', project_root)
print('JAX version:', jax.__version__)
print('Default backend:', jax.default_backend())
print('Devices:', jax.devices())

Project root: d:\@icog_projects\NGC-PC-Transformers
JAX version: 0.6.2
Default backend: cpu
Devices: [CpuDevice(id=0)]


In [4]:
def build_model_and_loader(seed=1234):
    dkey = random.PRNGKey(seed)

    seq_len = config.seq_len
    batch_size = config.batch_size
    n_embed = config.n_embed
    vocab_size = config.vocab_size

    data_loader = DataLoader(seq_len=seq_len, batch_size=batch_size)
    train_loader, _, _ = data_loader.load_and_prepare_data()

    model = NGCTransformer(
        dkey,
        batch_size=batch_size,
        seq_len=seq_len,
        n_embed=n_embed,
        vocab_size=vocab_size,
        n_layers=config.n_layers,
        n_heads=config.n_heads,
        T=config.n_iter,
        dt=1.0,
        tau_m=config.tau_m,
        act_fx=config.act_fx,
        eta=config.eta,
        dropout_rate=config.dropout_rate,
        exp_dir='exp',
        loadDir=None,
        pos_learnable=config.pos_learnable,
        optim_type=config.optim_type,
        wub=config.wub,
        wlb=config.wlb,
        model_name='ngc_transformer_bench'
    )
    return model, train_loader, vocab_size

def prepare_targets_onehot_flat_py(targets, vocab_size):
    return jax.nn.one_hot(targets, vocab_size, dtype=jnp.float32).reshape(-1, vocab_size)

@partial(jax.jit, static_argnums=(1,))
def prepare_targets_onehot_flat_jit(targets, vocab_size):
    return jax.nn.one_hot(targets, vocab_size, dtype=jnp.float32).reshape(-1, vocab_size)

def compute_batch_ce_ppl_py(y_mu, targets, vocab_size):
    y_pred = y_mu.reshape(-1, vocab_size)
    y_true = jax.nn.one_hot(targets.reshape(-1), vocab_size, dtype=y_pred.dtype)
    batch_nll = measure_CatNLL(y_pred, y_true)
    batch_ce_loss = batch_nll.mean()
    batch_ppl = jnp.exp(batch_ce_loss)
    return batch_ce_loss, batch_ppl

@partial(jax.jit, static_argnums=(2,))
def compute_batch_ce_ppl_jit(y_mu, targets, vocab_size):
    y_pred = y_mu.reshape(-1, vocab_size)
    y_true = jax.nn.one_hot(targets.reshape(-1), vocab_size, dtype=y_pred.dtype)
    batch_nll = measure_CatNLL(y_pred, y_true)
    batch_ce_loss = batch_nll.mean()
    batch_ppl = jnp.exp(batch_ce_loss)
    return batch_ce_loss, batch_ppl

In [ ]:
def benchmark_variant(
    use_jit_helpers=True,
    num_batches=30,
    adapt_synapses=True,
    seed=1234,
    max_seconds=None,
    warmup=True,
):
    model, train_loader, vocab_size = build_model_and_loader(seed=seed)

    prep_fn = prepare_targets_onehot_flat_jit if use_jit_helpers else prepare_targets_onehot_flat_py
    metric_fn = compute_batch_ce_ppl_jit if use_jit_helpers else compute_batch_ce_ppl_py

    iterator = iter(train_loader)
    warm_batch = next(iterator)
    warm_targets = jax.device_put(warm_batch[1][1])

    compile_seconds = 0.0
    if use_jit_helpers and warmup:
        t0 = time.perf_counter()
        warm_flat = prep_fn(warm_targets, vocab_size)
        jax.block_until_ready(warm_flat)

        dummy_y = jnp.zeros((warm_targets.shape[0], warm_targets.shape[1], vocab_size), dtype=jnp.float32)
        warm_ce, warm_ppl = metric_fn(dummy_y, warm_targets, vocab_size)
        jax.block_until_ready(warm_ce)
        jax.block_until_ready(warm_ppl)
        compile_seconds = time.perf_counter() - t0

    start = time.perf_counter()
    measured = 0
    tokens = 0

    while measured < num_batches:
        if max_seconds is not None and (time.perf_counter() - start) > max_seconds:
            break
        try:
            batch = next(iterator)
        except StopIteration:
            break

        inputs = jax.device_put(batch[0][1])
        targets = jax.device_put(batch[1][1])

        targets_flat = prep_fn(targets, vocab_size)
        y_mu_inf, y_mu, efe = model.process(obs=inputs, lab=targets_flat, adapt_synapses=adapt_synapses)

        ce, ppl = metric_fn(y_mu, targets, vocab_size)

        jax.block_until_ready(targets_flat)
        jax.block_until_ready(y_mu)
        jax.block_until_ready(ce)
        jax.block_until_ready(ppl)

        measured += 1
        tokens += int(targets.size)

    elapsed = time.perf_counter() - start
    return {
        'variant': 'jit_helpers' if use_jit_helpers else 'python_helpers',
        'batches': measured,
        'tokens': tokens,
        'adapt_synapses': adapt_synapses,
        'compile_seconds': compile_seconds,
        'steady_seconds': elapsed,
        'tokens_per_second': (tokens / elapsed) if elapsed > 0 else float('nan'),
        'timed_out': bool(max_seconds is not None and measured < num_batches),
    }

In [ ]:
NUM_BATCHES = 4
ADAPT_SYNAPSES = False
MAX_SECONDS = 90

baseline = benchmark_variant(
    use_jit_helpers=False,
    num_batches=NUM_BATCHES,
    adapt_synapses=ADAPT_SYNAPSES,
    seed=1234,
    max_seconds=MAX_SECONDS,
    warmup=False,
    )
jitted = benchmark_variant(
    use_jit_helpers=True,
    num_batches=NUM_BATCHES,
    adapt_synapses=ADAPT_SYNAPSES,
    seed=1234,
    max_seconds=MAX_SECONDS,
    warmup=True,
    )

df = pd.DataFrame([baseline, jitted])
speedup = baseline['steady_seconds'] / jitted['steady_seconds'] if jitted['steady_seconds'] > 0 else float('nan')

display(df)
print(f'Steady-state speedup (python_helpers / jit_helpers): {speedup:.3f}x')
print('Quick mode complete. For more stable numbers, raise NUM_BATCHES to 20-50 and ADAPT_SYNAPSES=True.')

,variant,batches,tokens,adapt_synapses,compile_seconds,steady_seconds,tokens_per_second
0,python_helpers,20,360,True,0.000000,569.481015,0.632155
1,jit_helpers,20,360,True,11.013207,610.795691,0.589395


Steady-state speedup (python_helpers / jit_helpers): 0.932x


## Decision Rule for PR

- If speedup is consistently above 1.05x across multiple runs, keep JIT helper changes.
- If speedup is ~1.00x (or slower), revert to the previous training helper implementation.
- Repeat the benchmark with `NUM_BATCHES` >= 50 for a more stable estimate.